# 03 · Diagnóstico de fallos del baseline

El baseline está congelado (notebook 02). Antes de mejorar nada hay que saber **por qué falla**, porque la causa decide qué arreglo la ataca: no sirve de nada afinar la búsqueda si el fallo es que el agente escribe en `cifra` un número que no toca.

Este notebook hace cuatro pasos sencillos:

1. **Dónde falla.** Lista las preguntas que fallan y en qué métrica. Es gratis: solo lee los CSV del baseline.
2. **Qué respondió el agente.** Repite únicamente esas preguntas para ver la respuesta (unos 2-3 ¢ cada una).
3. **Por qué falló.** Asigna a cada fallo una causa con reglas simples.
4. **Qué mejora la ataca.** Resume todo en un mapa *fallo → causa → mejora candidata*, que es una sección central del informe.

El baseline no se toca: los `eval_baseline_*.csv` quedan como están. El notebook escribe un fichero nuevo, `resultados/diagnostico_baseline.csv`.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "agente").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import pandas as pd

if not os.environ.get("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

from agente import datos, evaluadores, interfaz, trazas

RUTAS = {"propio": "golden/golden_set.jsonl",
         "oficial": "golden/oficial_20.jsonl",
         "huecos": "golden/huecos_humo.jsonl"}
GOLDEN = {nombre: evaluadores.cargar_golden(RAIZ / ruta) for nombre, ruta in RUTAS.items()}
ITEMS = {g["id"]: g for items in GOLDEN.values() for g in items}     # id -> pregunta del golden
BASELINE = {nombre: pd.read_csv(RAIZ / f"resultados/eval_baseline_{nombre}.csv") for nombre in RUTAS}

print({nombre: len(df) for nombre, df in BASELINE.items()}, "filas del baseline cargadas")

## Paso 1 — Dónde falla

El evaluador da, para cada pregunta, cuatro señales:

| Columna | Qué mide |
|---|---|
| `cifra_ok` | La cifra del agente coincide con XBRL (tolerancia del 1 %) |
| `cita_ok` | La cita existe en el corpus y respalda lo que dice |
| `tool_ok` | Pasó por la herramienta que tocaba |
| `recall` | El ancla del golden entra en el top-5 de la búsqueda |

Un hueco significa "no aplica" (una numérica no tiene cita, una extractiva no tiene cifra). Aquí solo contamos los `False` y los errores.

Un detalle: `recall` mide el buscador aparte del agente. Puede fallar y aun así la respuesta salir bien, así que no es lo mismo que un fallo de la pregunta.

In [ ]:
filas = []
for nombre, df in BASELINE.items():
    for r in df.itertuples():
        filas.append({
            "golden": nombre, "id": r.id, "familia": r.familia,
            "falla_cifra": r.cifra_ok == False,        # noqa: E712  (NaN == False es False)
            "falla_cita": r.cita_ok == False,          # noqa: E712
            "falla_retrieval": r.recall == False,      # noqa: E712
            "error": bool(pd.notna(r.error)),
        })
tabla = pd.DataFrame(filas)
MOTIVOS = ["falla_cifra", "falla_cita", "falla_retrieval", "error"]
fallos = tabla[tabla[MOTIVOS].any(axis=1)].reset_index(drop=True)

print(f"{len(fallos)} preguntas con algún fallo, de {len(tabla)}:")
display(fallos)

print("Cuántos fallos de cada tipo, por golden y familia:")
display(tabla.groupby(["golden", "familia"])[MOTIVOS].sum())

## Paso 2 — Qué respondió el agente

Los CSV del baseline solo guardan `True`/`False`: dicen **que** algo falló, no **qué** respondió el agente. Para saberlo hay que verlo, así que repetimos solo las preguntas que fallaron en cifra o en cita (cada una en un hilo nuevo, aislado) y miramos su respuesta estructurada: la cifra que puso, el concepto XBRL, la fuente y el fragmento citado.

Tres advertencias:

- **Necesita la clave** y cuesta unos 2-3 ¢ por pregunta.
- **El modelo no es determinista.** Un fallo puede no reproducirse en la repetición. No es un error del notebook: es ruido, y se cuenta como tal en el Paso 3.
- Si el proveedor limita las peticiones, la repetición espera 30 s y reintenta una vez.

In [ ]:
# Se repiten las que fallaron en cifra o en cita (o dieron error). Las que solo fallan en
# retrieval respondieron bien: no hace falta gastar una llamada para verlas.
A_REPETIR = fallos[fallos.falla_cifra | fallos.falla_cita | fallos.error]["id"].tolist()
print(f"{len(A_REPETIR)} preguntas a repetir (≈ {len(A_REPETIR) * 2.5:.0f} ¢): {A_REPETIR}\n")


def repetir(ident):
    """Una repetición del agente, con un reintento si el proveedor limita las peticiones."""
    for intento in (1, 2):
        try:
            return interfaz.responder(ITEMS[ident]["pregunta"], thread_id=f"diag-{ident}-{intento}")
        except Exception as e:
            if intento == 2:
                raise
            print(f"   {type(e).__name__}: espero 30 s y reintento")
            time.sleep(30)


REPETICIONES = {}
for ident in A_REPETIR:
    g = ITEMS[ident]
    print(f"[{ident}] {g['pregunta'][:90]}")
    try:
        r = repetir(ident)
    except Exception as e:
        print(f"   ERROR: {type(e).__name__}: {str(e)[:100]}\n")
        continue
    REPETICIONES[ident] = r
    s = r.get("structured_response")
    if s is None:
        print("   el agente no devolvió respuesta estructurada\n")
        continue
    print(f"   esperada: {g.get('cifra_esperada')} ({g.get('concept_xbrl')})")
    print(f"   agente  : cifra={s.cifra} · concepto={s.concept_xbrl} · fuente={s.fuente} · chunk={s.chunk_id}")
    print(f"   herramientas: {trazas.herramientas_usadas(r)}\n")

## Paso 3 — Por qué falló

Con la respuesta delante, cada fallo se clasifica con reglas simples, en este orden:

| Causa | Cómo se detecta | Mejora que la ataca |
|---|---|---|
| **cifra: variación / valor del ejercicio anterior** | La `cifra` del agente coincide con la diferencia, el porcentaje o el valor del año anterior, no con el del ejercicio pedido | Fijar la convención en el prompt + middleware de cifras |
| **cifra: escala equivocada** | Coincide con la esperada multiplicando por 10³, 10⁶ o 10⁹ | Middleware de cifras (normaliza unidades) |
| **cifra: sin cifra / sin respaldo en XBRL** | No hay cifra, o no coincide con ningún valor conocido | Middleware de cifras contra XBRL |
| **retrieval** | La cita falla y el ancla tampoco entraba en el top-5 de la búsqueda | Búsqueda híbrida en `search_filings` |
| **cita** | La cita falla aunque la búsqueda sí encontraba el ancla | Prompt: citar el `chunk_id` del fragmento usado |
| **enrutado** | No usó la herramienta esperada | Docstrings y prompt |
| **no se reproduce** | En la repetición ya no falla | Ninguna: es ruido del modelo |

**Hipótesis que esta celda comprueba en lugar de suponer.** En las comparativas, el prompt dice "calcula tú la variación" y el golden espera en `cifra` el valor del último ejercicio. Si el agente escribe la variación en `cifra`, el evaluador marca fallo aunque el razonamiento sea bueno. Sería un desajuste de convención, no un error de comprensión, y se arregla distinto que un fallo de búsqueda.

In [ ]:
XBRL = datos.cargar_xbrl()


def valor(ticker, ejercicio, concepto):
    f = XBRL[(XBRL.ticker == ticker) & (XBRL.fiscal_year == ejercicio) & (XBRL.concept == concepto)]
    return float(f.iloc[0].value) if len(f) else None


def que_cifra_dio(g, s):
    """¿Qué número puso el agente en `cifra`, comparado con el esperado?"""
    if s.cifra is None:
        return "sin cifra"
    actual = g["cifra_esperada"]
    previo = valor(g["ticker"], g["fiscal_year"] - 1, g["concept_xbrl"])
    if previo:
        candidatos = {"valor del ejercicio anterior": previo,
                      "variación absoluta": actual - previo,
                      "variación en %": (actual / previo - 1) * 100,
                      "variación en fracción": actual / previo - 1}
        for nombre, v in candidatos.items():
            if evaluadores.cuadra(s.cifra, v):
                return nombre
    for k in (1e3, 1e6, 1e9):
        if evaluadores.cuadra(s.cifra * k, actual):
            return "escala equivocada"
    return "sin respaldo en XBRL"


def causas(g, r, fila):
    """Causas del fallo en esta repetición; lista vacía si ya no se reproduce."""
    s = r["structured_response"]
    lista = []
    if evaluadores.cifra_coincide_xbrl(g, r) is False:
        lista.append("cifra: " + que_cifra_dio(g, s))
    if evaluadores.cita_correcta(g, r) is False:
        lista.append("retrieval" if fila.falla_retrieval else "cita")
    if evaluadores.uso_la_tool_correcta(g, r) is False:
        lista.append("enrutado")
    return lista


def mejora(causa):
    if causa.startswith("cifra: variación") or causa == "cifra: valor del ejercicio anterior":
        return "Convención en el prompt (cifra = valor del ejercicio; la variación, en la prosa) + middleware"
    if causa.startswith("cifra"):
        return "Middleware de cifras contra XBRL"
    if causa.startswith("retrieval"):
        return "Búsqueda híbrida en search_filings"
    if causa == "cita":
        return "Prompt: citar el chunk_id del fragmento usado"
    if causa == "enrutado":
        return "Docstrings y prompt"
    if causa == "sin respuesta estructurada":
        return "Reintento y arreglo del formato de salida"
    return "Ninguna: ruido del modelo"


por_id = fallos.set_index("id")
filas_diag = []
for ident, r in REPETICIONES.items():
    if r.get("structured_response") is None:
        lista = ["sin respuesta estructurada"]
    else:
        lista = causas(ITEMS[ident], r, por_id.loc[ident]) or ["no se reproduce"]
    for c in lista:
        filas_diag.append({"golden": por_id.loc[ident, "golden"], "id": ident,
                           "familia": ITEMS[ident]["familia"], "causa": c, "mejora candidata": mejora(c)})

# Las que solo fallaban en retrieval (la respuesta salió bien) no se repitieron: se anotan tal cual.
solo_retrieval = fallos[fallos.falla_retrieval & ~fallos.falla_cifra & ~fallos.falla_cita & ~fallos.error]
for r in solo_retrieval.itertuples():
    filas_diag.append({"golden": r.golden, "id": r.id, "familia": r.familia,
                       "causa": "retrieval (la respuesta salió bien)",
                       "mejora candidata": mejora("retrieval")})

diagnostico = pd.DataFrame(filas_diag).sort_values(["golden", "id"]).reset_index(drop=True)
display(diagnostico)

## Paso 4 — El mapa fallo → causa → mejora

Se agrupa el diagnóstico por **mejora candidata** (varias causas parecidas comparten arreglo) y se cuenta cuántas preguntas afecta cada una. Se lee de arriba abajo: la primera fila es la mejora que más preguntas ataca, y por tanto la que más rinde. Ese orden decide qué entra en los notebooks 04 (guardrails) y 05 (mejoras medidas).

Un recordatorio para leer los números: es una repetición por pregunta con un modelo no determinista. Un fallo que "no se reproduce" no es un fallo del sistema sino ruido, y conviene contarlo aparte en lugar de esconderlo.

In [ ]:
resumen = (diagnostico
           .groupby("mejora candidata")
           .agg(preguntas=("id", "nunique"),
                causas=("causa", lambda s: ", ".join(sorted(set(s)))),
                ids=("id", lambda s: ", ".join(sorted(set(s)))))
           .sort_values("preguntas", ascending=False)
           .reset_index())
display(resumen)

diagnostico.to_csv(RAIZ / "resultados/diagnostico_baseline.csv", index=False)
print("Guardado: resultados/diagnostico_baseline.csv")

total = diagnostico["id"].nunique()
de_cifra = diagnostico[diagnostico.causa.str.startswith("cifra")]["id"].nunique()
de_retrieval = diagnostico[diagnostico.causa.str.startswith("retrieval")]["id"].nunique()
print(f"\nDe {total} preguntas con fallo: {de_cifra} lo son por la cifra y {de_retrieval} por retrieval.")
if de_cifra > de_retrieval:
    print("→ La cifra pesa más que la búsqueda: el middleware y la convención del prompt van antes que "
          "cualquier mejora de retrieval.")
else:
    print("→ El retrieval pesa tanto como la cifra o más: la búsqueda híbrida va a la cabeza.")

## Qué sigue

- Las causas de **cifra** son el trabajo del **notebook 04**: el middleware que contrasta lo que afirma el agente con XBRL y devuelve el desajuste al modelo, junto con el límite de llamadas por invocación.
- Las causas de **retrieval y de cita** entran en el **notebook 05** como mejoras medidas una a una (búsqueda híbrida, cambios de prompt), cada una con su fila en la tabla y su coste.
- Lo que aparezca como **ruido** no se arregla: se cuenta y se explica en el informe.

Cuando el notebook esté ejecutado, `resultados/diagnostico_baseline.csv` es la evidencia de la tabla *fallo → causa → mejora* del informe.

In [ ]:
print("NOTEBOOK 03 COMPLETADO — fallos del baseline diagnosticados.")
print("Fichero escrito: resultados/diagnostico_baseline.csv")
print("Siguiente: notebook 04 (guardrails: middleware de cifras y límite de llamadas)")